In [1]:
import pandas as pd

df = pd.read_parquet("../data/processed/crimes_enriched.parquet")
d = pd.to_datetime(df["Date"])

df["hour"] = d.dt.hour
df["day_of_week"] = d.dt.dayofweek          # 0 = Monday ... 6 = Sunday
df["month"] = d.dt.month
df["year"] = d.dt.year
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

def get_season(m):
    if m in (12, 1, 2):  return "WINTER"
    if m in (3, 4, 5):   return "SPRING"
    if m in (6, 7, 8):   return "SUMMER"
    return "FALL"

df["season"] = df["month"].apply(get_season)

def get_time_of_day(h):
    if h < 6:    return "NIGHT"        # 00:00 - 05:59
    if h < 12:   return "MORNING"      # 06:00 - 11:59
    if h < 18:   return "AFTERNOON"    # 12:00 - 17:59
    return "EVENING"                   # 18:00 - 23:59

df["time_of_day"] = df["hour"].apply(get_time_of_day)

print(df.shape)
print(df["season"].value_counts())
print(df["time_of_day"].value_counts())
print("Weekend share:", round(df["is_weekend"].mean() * 100, 1), "%")
print(df[["Date", "hour", "day_of_week", "month", "season", "is_weekend", "time_of_day"]].head())

(921013, 40)
season
SUMMER    268792
SPRING    247803
WINTER    205960
FALL      198458
Name: count, dtype: int64
time_of_day
AFTERNOON    286944
EVENING      260379
MORNING      191295
NIGHT        182395
Name: count, dtype: int64
Weekend share: 28.8 %
        Date  hour  day_of_week  month season  is_weekend time_of_day
0 2026-09-10     0            3      9   FALL           0       NIGHT
1 2026-09-10     0            3      9   FALL           0       NIGHT
2 2026-09-10     0            3      9   FALL           0       NIGHT
3 2026-09-10     0            3      9   FALL           0       NIGHT
4 2026-09-10     0            3      9   FALL           0       NIGHT


In [2]:
# the merge left two copies of the area number: confirm they match, then keep one
print("Area columns equal:", (df["Community Area"] == df["community_area"]).all())
df = df.drop(columns=["Community Area"])

df = df.rename(columns={
    "Date": "datetime", "Block": "block", "IUCR": "iucr",
    "Primary Type": "primary_type", "Description": "description",
    "Location Description": "location_description",
    "Arrest": "arrest", "Domestic": "domestic", "Beat": "beat",
    "District": "district", "Ward": "ward", "FBI Code": "fbi_code",
    "Latitude": "latitude", "Longitude": "longitude",
})

df["district"] = df["district"].astype(int)
df["has_coordinates"] = df["latitude"].notna().astype(int)

print(df.shape)
print(df.columns.tolist())
print("Rows with coordinates:", round(df["has_coordinates"].mean() * 100, 1), "%")
print(df[["district", "beat", "community_area", "arrest", "domestic"]].dtypes)

df.to_parquet("../data/processed/crimes_features.parquet", index=False)
print("Saved", df.shape)

Area columns equal: True
(921013, 40)
['datetime', 'block', 'iucr', 'primary_type', 'description', 'location_description', 'arrest', 'domestic', 'beat', 'district', 'ward', 'fbi_code', 'latitude', 'longitude', 'location_group', 'is_violent', 'crime_group', 'date', 'community_area', 'area_name', 'pct_crowded', 'poverty_rate', 'unemployment', 'pct_no_diploma', 'pct_dependent', 'per_capita_income', 'hardship_index', 'population', 'temperature', 'rainfall', 'is_holiday', 'is_new_year', 'hour', 'day_of_week', 'month', 'year', 'is_weekend', 'season', 'time_of_day', 'has_coordinates']
Rows with coordinates: 99.3 %
district            int64
beat                  str
community_area    float64
arrest              int64
domestic            int64
dtype: object
Saved (921013, 40)


In [3]:
df["community_area"] = df["community_area"].astype(int)

# 1. missing values (only these columns are allowed to have any)
miss = df.isna().sum()
print("Columns with missing values:\n", miss[miss > 0], "\n")

# 2. dtypes of the key columns
print(df[["community_area", "district", "beat", "hour", "day_of_week", "month", "year"]].dtypes, "\n")

# 3. value ranges
print("hour:", df["hour"].min(), "-", df["hour"].max())
print("community_area:", df["community_area"].min(), "-", df["community_area"].max(), "| unique:", df["community_area"].nunique())
print("districts:", df["district"].nunique())

# 4. the target and flag shares
print("\nViolent share:", round(df["is_violent"].mean() * 100, 1), "%")
print("Arrest share:", round(df["arrest"].mean() * 100, 1), "%")
print("Holiday rows:", round(df["is_holiday"].mean() * 100, 1), "%")

# 5. rows per year and a final shape
print("\n", df.groupby("year").size())

df.to_parquet("../data/processed/crimes_features.parquet", index=False)
print("\nSaved", df.shape)

Columns with missing values:
 location_description    4702
ward                       3
latitude                6314
longitude               6314
dtype: int64 

community_area    int64
district          int64
beat                str
hour              int32
day_of_week       int32
month             int32
year              int32
dtype: object 

hour: 0 - 23
community_area: 1 - 77 | unique: 77
districts: 23

Violent share: 33.4 %
Arrest share: 14.3 %
Holiday rows: 3.8 %

 year
2023    262757
2024    259316
2025    237820
2026    161120
dtype: int64

Saved (921013, 40)


In [1]:
import pandas as pd

df = pd.read_parquet("../data/processed/crimes_features.parquet",
                     columns=["datetime", "primary_type", "hour"])

exact0 = df["datetime"].dt.hour.eq(0) & df["datetime"].dt.minute.eq(0)
exact12 = df["datetime"].dt.hour.eq(12) & df["datetime"].dt.minute.eq(0)

print("Exactly 00:00:", f"{exact0.sum():,}", "| Exactly 12:00:", f"{exact12.sum():,}")
print("Share of hour-0 rows that are exactly 00:00:", round(exact0.sum() / (df["hour"] == 0).sum() * 100, 1), "%")
print("Share of hour-12 rows that are exactly 12:00:", round(exact12.sum() / (df["hour"] == 12).sum() * 100, 1), "%")
print("\nCrime types at exactly 00:00:")
print(df[exact0]["primary_type"].value_counts().head(6))
print("\nShare of 00:00 rows falling on the 1st of a month:",
      round(df[exact0]["datetime"].dt.day.eq(1).mean() * 100, 1), "%")

Exactly 00:00: 33,652 | Exactly 12:00: 21,300
Share of hour-0 rows that are exactly 00:00: 52.4 %
Share of hour-12 rows that are exactly 12:00: 40.6 %

Crime types at exactly 00:00:
primary_type
DECEPTIVE PRACTICE     6997
THEFT                  6981
CRIMINAL DAMAGE        3746
MOTOR VEHICLE THEFT    3036
OTHER OFFENSE          2920
BATTERY                2838
Name: count, dtype: int64

Share of 00:00 rows falling on the 1st of a month: 9.6 %


In [2]:
import pandas as pd

df = pd.read_parquet("../data/processed/crimes_features.parquet")
t = df["datetime"]
exact = (t.dt.minute == 0) & (t.dt.second == 0)
df["time_unknown"] = (exact & t.dt.hour.isin([0, 12])).astype(int)

print("Flagged rows:", f"{df['time_unknown'].sum():,}",
      "=", round(df["time_unknown"].mean() * 100, 1), "% of all rows")

by_hour = pd.DataFrame({
    "all_rows": df.groupby("hour").size(),
    "pct_exactly_on_the_hour": (exact.groupby(df["hour"]).mean() * 100).round(1),
    "without_flagged": df[df["time_unknown"] == 0].groupby("hour").size(),
})
print(by_hour)

df.to_parquet("../data/processed/crimes_features.parquet", index=False)
print("Saved", df.shape)

Flagged rows: 54,952 = 6.0 % of all rows
      all_rows  pct_exactly_on_the_hour  without_flagged
hour                                                    
0        64252                     52.4            30600
1        30565                     30.1            30565
2        27082                     30.6            27082
3        23889                     31.0            23889
4        19520                     28.3            19520
5        17087                     30.3            17087
6        18158                     31.5            18158
7        23463                     31.7            23463
8        31634                     35.1            31634
9        38124                     41.1            38124
10       39426                     33.0            39426
11       40490                     26.0            40490
12       52449                     40.6            31149
13       41614                     29.0            41614
14       44147                     29.7        